# 1. Data acquisition, exploration, and preprocessing

**Research question:** Do Random Forest and XGBoost improve on simple prediction and portfolio baselines?

Use 2015–2021 for training and 2022–2023 for validation. Preserve 2024–2025 for the assignment test and **2026 as a separate competition holdout**. No 2026 performance is explored here.

Run the three notebooks in numerical order, each in a fresh kernel. Install `requirements.txt` and select that environment. Shared, tested implementations are in `stockml/`; retain that folder with the notebooks. Normal runs download real data. The automated smoke runner explicitly uses deterministic synthetic data and reduced models.

In [ ]:
from pathlib import Path
import os
import sys

# Launch from the repository root. All handoffs are explicit disk artifacts.
PROJECT = Path.cwd()
if not (PROJECT / 'stockml').is_dir():
    raise RuntimeError('Open this notebook with the repository root as the working directory.')
sys.path.insert(0, str(PROJECT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT / '.cache' / 'matplotlib'))
import pandas as pd
from IPython.display import display, Markdown
from stockml.common import root, smoke, load_json

pd.set_option('display.max_columns', 20)
print('Artifacts:', root())
print('SYNTHETIC SMOKE TEST — NOT FINANCIAL RESULTS' if smoke() else 'REAL-DATA EXPERIMENT')

## Acquire and lock a reproducible snapshot

`AS_OF=None` uses the latest available session through yesterday in New York, capped at December 2026. The current day is conservatively excluded. A cached snapshot is verified and reused, not silently refreshed. To intentionally acquire another snapshot, use a different `STOCK_ML_ARTIFACTS` directory; changing snapshots after inspecting results does not restore holdout independence.

All OHLC fields use `auto_adjust=True`. This supports a consistent corporate-action-adjusted fractional-unit simulation, not literal historical share counts or a point-in-time execution database. VIX volume may be zero. Stocks must have positive volume. VIX-only dates outside SPY's equity calendar are excluded and recorded in metadata. Missing equity-session observations or invalid prices stop the workflow rather than being filled.

In [ ]:
from stockml.data import prepare_data
AS_OF = None  # Optional explicit completed calendar date, e.g. '2026-10-07'.
metadata = prepare_data(as_of=AS_OF)
display(pd.Series({key: metadata[key] for key in ['downloaded_at_utc', 'last_completed_session', 'synthetic', 'adjustment', 'excluded_vix_non_equity_dates']}))
display(pd.Series(metadata['versions'], name='installed_version'))

## Development-only exploratory analysis

Prices and returns can have changing volatility, heavy tails, and strong cross-stock dependence. Predictable volatility does not imply predictable return direction. The eight retrospectively chosen stocks also introduce survivorship and selection bias. Interpret the plots as descriptive evidence, not proof that a profitable pattern exists.

In [ ]:
from stockml.reporting import plot_eda
plot_eda()

## Feature and target definitions

All features use observations available at the signal close. Returns are fractions (0.01 means 1%). Rolling standard deviations use `ddof=1`. RSI uses Wilder-style exponential smoothing with `alpha=1/14`, `adjust=False`, and a 14-observation warm-up. MACD is the 12-span minus 26-span exponential moving average, divided by the current close. No scaling or imputation is fitted here.

| Category | Predictors |
|---|---|
| Momentum | 1-, 5-, 10-, 20-session returns |
| Trend | Close / 5- and 20-session mean close minus one |
| Volatility | 5- and 20-session standard deviation of daily returns |
| Volume | Volume percentage change; volume / 20-session mean volume |
| Technical | RSI-14; normalized MACD |
| Market context | SPY 1- and 5-session returns; VIX level and percentage change |
| Relative performance | Stock 5-session return minus SPY 5-session return |

The return target is $P_{t+5}/P_t-1$. Future risk is the sample standard deviation of returns on sessions $t+1,\ldots,t+5$. Both require all five future observations. Keep a `target_end` date for leakage checks. High-risk thresholds are fitted in Notebook 2, never here.

Rolling features are calculated continuously across year boundaries. Historical outcomes extending into 2026 are blanked before export. Inference rows with unknown future outcomes remain available.

In [ ]:
from stockml.common import FEATURES, read_table
features = read_table('history_features')
outcomes = read_table('history_outcomes')
assert len(FEATURES) == 17
assert not features.duplicated(['date', 'ticker']).any()
assert outcomes.target_end.dropna().max() < pd.Timestamp('2026-01-01')
display(features[features.date < '2024-01-01'].head())
display(outcomes[outcomes.date < '2024-01-01'].head())
print('Saved historical and holdout features, outcomes, and prices separately; SHA-256 fingerprints recorded.')

## Handoff

`data/history_{features,outcomes,prices}.parquet` contains the historical experiment inputs. `data/holdout_{features,outcomes,prices}.parquet` contains 2026 inputs. `data/metadata.json` records the snapshot, definitions, versions, and fingerprints.

Do not inspect holdout outcomes to modify features. Continue with Notebook 2.